# EP01 — Portfolio Correlation & Variance
**Quantifaya · Classical Quantitative Finance Series · Episode 1**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Godwin-88/quantifire-web/blob/main/public/notebooks/ep01-correlation-matters.ipynb)

> **Learning objective:** Understand why correlation — not individual asset returns — drives portfolio risk reduction, and implement this from scratch in Python.

**Companion post:** [quantifaya.com/blog/ep01-why-correlation-matters-more-than-returns](https://quantifaya.com/blog/ep01-why-correlation-matters-more-than-returns)

---
*Quantifaya research notebooks are provided for educational purposes only. Nothing here constitutes financial advice.*

## Learning Objectives

By the end of this notebook you will be able to:

- **Derive** portfolio variance from first principles using the covariance matrix formulation $\sigma^2_p = w^T \Sigma w$
- **Fetch and clean** multi-asset price data from Yahoo Finance using `yfinance`
- **Compute** log returns, annualised volatility, and correlation matrices from raw price series
- **Visualise** correlation structure with interactive Plotly heatmaps, dendrograms, and rolling-window charts
- **Quantify** the diversification benefit mathematically: how low correlation reduces portfolio variance below the weighted-average of individual variances
- **Stress-test** a portfolio through a simulated crisis period (COVID-19 Feb–Apr 2020) and observe how correlations shift
- **Implement** Marginal Contribution to Risk (MCTR) to identify which asset is the dominant risk driver
- **Simulate** the Efficient Frontier via Monte Carlo and locate the minimum-variance and maximum-Sharpe portfolios
- **Build** a full Hierarchical Risk Parity (HRP) optimiser from scratch following López de Prado (2016)
- **Compare** Equal Weight, Min Variance, Max Sharpe, and HRP portfolios across return, volatility, Sharpe, and drawdown metrics

## Mathematical Prerequisites

### Log Returns

We use log returns throughout for their additive time-aggregation property:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

### Covariance

The covariance between assets $i$ and $j$ measures how their returns move together:

$$\text{Cov}(R_i, R_j) = \mathbb{E}\left[(R_i - \mu_i)(R_j - \mu_j)\right]$$

### Pearson Correlation

The standardised form of covariance, bounded in $[-1, +1]$:

$$\rho_{ij} = \frac{\text{Cov}(R_i, R_j)}{\sigma_i \, \sigma_j}$$

### Portfolio Variance — 2-Asset Case

$$\sigma^2_p = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \sigma_1 \sigma_2 \rho_{12}$$

The key insight: the final term can be made *negative* when $\rho_{12} < 0$, reducing total variance below the weighted sum of individual variances.

### Portfolio Variance — N-Asset Case (Matrix Form)

$$\sigma^2_p = w^T \Sigma w = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \sigma_{ij}$$

where $\Sigma$ is the $N \times N$ covariance matrix and $w$ is the weight vector.

### Diversification Benefit

$$\Delta\sigma^2 = \sum_{i=1}^{N} w_i^2 \sigma_i^2 - \sigma^2_p = -\sum_{i \ne j} w_i w_j \sigma_{ij}$$

A positive $\Delta\sigma^2$ means the portfolio variance is lower than a naive sum-of-variances would suggest — this is the **diversification benefit**.

### Marginal Contribution to Risk (MCTR)

The partial derivative of portfolio volatility with respect to weight $w_i$:

$$\text{MCTR}_i = \frac{\partial \sigma_p}{\partial w_i} = \frac{(\Sigma w)_i}{\sigma_p}$$

The **component risk** (percent contribution) for asset $i$ is:

$$\text{CR}_i = w_i \cdot \text{MCTR}_i \;/\; \sigma_p$$

### Ledoit–Wolf Shrinkage

The Ledoit–Wolf estimator shrinks the sample covariance matrix $S$ towards a structured target $T$ (e.g., constant-correlation matrix):

$$\hat{\Sigma} = (1 - \alpha)\, S + \alpha\, T$$

where the optimal shrinkage intensity $\alpha^*$ minimises the expected Frobenius loss $\mathbb{E}\left[\|\hat{\Sigma} - \Sigma\|_F^2\right]$ under an asymptotic analytical formula (Ledoit & Wolf, 2004).

### HRP — Recursive Bisection Weight

At each bisection step, the HRP algorithm splits a cluster $C$ into left $C_L$ and right $C_R$ sub-clusters. The weight allocated to $C_L$ is:

$$\tilde{w}_{C_L} = 1 - \frac{\tilde{V}_{C_L}}{\tilde{V}_{C_L} + \tilde{V}_{C_R}}$$

where $\tilde{V}_{C_k} = \left(\sum_{i \in C_k} \frac{1}{\sigma_i^2}\right)^{-1}$ is the inverse-variance portfolio variance of sub-cluster $k$.

In [ ]:
# ▶ Run this cell first — installs all required packages
!pip install yfinance pandas numpy scipy plotly --quiet

---
## ▶ STEP 1 — Configure Your Analysis

Edit the `CONFIG` dictionary in the next cell to customise the asset universe, date range, and portfolio weights. All downstream cells will automatically adapt.

In [ ]:
# ═══════════════════════════════════════════════════════════
# USER CONFIGURATION — Edit this cell to customise the analysis
# ═══════════════════════════════════════════════════════════

CONFIG: dict = {
    # Asset universe — replace with any Yahoo Finance tickers
    'tickers': ['SPY', 'TLT', 'GLD', 'VNQ', 'EEM'],
    'asset_names': {
        'SPY': 'S&P 500',
        'TLT': '20Y Treasury',
        'GLD': 'Gold',
        'VNQ': 'REITs',
        'EEM': 'Emerging Mkts',
    },

    # Date range
    'start_date': '2015-01-01',
    'end_date':   '2024-12-31',

    # Portfolio weights (must sum to 1.0 — set to None for equal weight)
    'weights': None,  # e.g., [0.40, 0.30, 0.10, 0.10, 0.10]

    # Annualisation
    'trading_days_per_year': 252,

    # Crisis period for stress testing
    'crisis_start': '2020-02-01',
    'crisis_end':   '2020-04-30',

    # HRP linkage method
    'linkage_method': 'single',   # 'single' | 'complete' | 'average' | 'ward'

    # Plotting
    'template': 'plotly_dark',
    'color_palette': ['#ef4444', '#3b82f6', '#f59e0b', '#10b981', '#8b5cf6'],
}

print("✅ Configuration loaded")
print(f"   Tickers : {CONFIG['tickers']}")
print(f"   Period  : {CONFIG['start_date']} → {CONFIG['end_date']}")
print(f"   Weights : {'equal-weight' if CONFIG['weights'] is None else CONFIG['weights']}")

In [ ]:
# ─────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

from __future__ import annotations
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.covariance import LedoitWolf
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import plotly.express as px

print("✅ All imports successful")
print(f"   pandas {pd.__version__} | numpy {np.__version__} | plotly available")

---
## ▶ STEP 2 — Fetch Market Data

We download **adjusted closing prices** from Yahoo Finance via `yfinance`. Adjusted prices account for dividends and stock splits, making them suitable for return calculation.

In [ ]:
# ─────────────────────────────────────────────
# Data Fetching
# ─────────────────────────────────────────────

def fetch_market_data(
    tickers: List[str],
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """Download adjusted closing prices from Yahoo Finance.

    Uses yfinance batch download for efficiency. Returns a DataFrame indexed
    by date with one column per ticker containing adjusted closing prices.

    Args:
        tickers: List of Yahoo Finance ticker symbols (e.g. ['SPY', 'TLT']).
        start_date: Inclusive start date in 'YYYY-MM-DD' format.
        end_date: Inclusive end date in 'YYYY-MM-DD' format.

    Returns:
        DataFrame of shape (T, N) where T is the number of trading days
        and N is the number of tickers. Index is a DatetimeIndex.

    Raises:
        ValueError: If no data is returned for any ticker.
    """
    print(f"Downloading data for {tickers} from {start_date} to {end_date}...")
    raw = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
    )

    if raw.empty:
        raise ValueError("yfinance returned an empty DataFrame. Check tickers and date range.")

    # Extract 'Close' level from MultiIndex columns when multiple tickers
    if isinstance(raw.columns, pd.MultiIndex):
        prices = raw['Close']
    else:
        prices = raw[['Close']]
        prices.columns = tickers

    prices = prices[tickers]  # enforce column order
    prices.index = pd.to_datetime(prices.index)
    return prices


# ── Fetch ──────────────────────────────────────
prices_raw: pd.DataFrame = fetch_market_data(
    tickers=CONFIG['tickers'],
    start_date=CONFIG['start_date'],
    end_date=CONFIG['end_date'],
)

print(f"\n✅ Data fetched: {prices_raw.shape[0]} rows × {prices_raw.shape[1]} columns")
print(f"   Date range : {prices_raw.index[0].date()} → {prices_raw.index[-1].date()}")
print("\nFirst 5 rows:")
display(prices_raw.head())
print("\nDataFrame info:")
prices_raw.info()

In [ ]:
# ─────────────────────────────────────────────
# Data Quality Checks
# ─────────────────────────────────────────────

def clean_price_data(prices: pd.DataFrame, max_consec_ffill: int = 5) -> pd.DataFrame:
    """Perform quality checks and clean raw price data.

    Steps applied in order:
      1. Report missingness per ticker.
      2. Forward-fill gaps of up to ``max_consec_ffill`` consecutive NaNs
         (e.g. public holidays where only some markets closed).
      3. Drop any remaining rows that still contain NaN values.

    Args:
        prices: Raw price DataFrame as returned by ``fetch_market_data``.
        max_consec_ffill: Maximum number of consecutive NaNs to forward-fill.
            Gaps longer than this are left as NaN and will be dropped.

    Returns:
        Cleaned price DataFrame with no NaN values.
    """
    print("=" * 50)
    print("DATA QUALITY REPORT")
    print("=" * 50)

    total_rows = len(prices)
    report_rows = []

    for col in prices.columns:
        n_missing = prices[col].isna().sum()
        pct_complete = 100 * (1 - n_missing / total_rows)
        report_rows.append({
            'Ticker': col,
            'Total Rows': total_rows,
            'Missing': n_missing,
            '% Complete': f"{pct_complete:.1f}%",
        })

    report_df = pd.DataFrame(report_rows).set_index('Ticker')
    display(report_df)

    # Forward-fill short gaps
    prices_filled = prices.fillna(method='ffill', limit=max_consec_ffill)
    n_ffilled = prices.isna().sum().sum() - prices_filled.isna().sum().sum()
    if n_ffilled > 0:
        print(f"\n  Forward-filled {n_ffilled} cell(s) (max gap = {max_consec_ffill} days)")

    # Drop rows with any remaining NaN
    prices_clean = prices_filled.dropna()
    n_dropped = total_rows - len(prices_clean)
    if n_dropped > 0:
        print(f"  Dropped {n_dropped} row(s) with remaining NaN values")

    print(f"\n✅ Clean dataset: {prices_clean.shape[0]} rows × {prices_clean.shape[1]} columns")
    return prices_clean


prices: pd.DataFrame = clean_price_data(prices_raw)

---
## ▶ STEP 3 — Calculate Log Returns

We compute **log (continuously compounded) returns**:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns are preferred over simple returns for quantitative work because:
- They are **time-additive**: $r_{t_1 \to t_3} = r_{t_1 \to t_2} + r_{t_2 \to t_3}$
- They are more **symmetrically distributed** (closer to Gaussian)
- They map to the **continuous compounding** framework used in derivatives pricing

In [ ]:
# ─────────────────────────────────────────────
# Log Returns
# ─────────────────────────────────────────────

def calculate_log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Compute daily log returns from a price DataFrame.

    Applies the formula $r_t = \\ln(P_t / P_{t-1})$ element-wise and drops
    the first row (which becomes NaN after the shift).

    Args:
        prices: DataFrame of adjusted closing prices, shape (T, N).

    Returns:
        DataFrame of log returns, shape (T-1, N), with the same column names.
    """
    log_returns = np.log(prices / prices.shift(1)).dropna()
    return log_returns


returns: pd.DataFrame = calculate_log_returns(prices)

# ── Summary statistics ──────────────────────
ann = CONFIG['trading_days_per_year']
names = CONFIG['asset_names']

stats = pd.DataFrame({
    'Asset': [names.get(c, c) for c in returns.columns],
    'Ann. Return (%)': (returns.mean() * ann * 100).round(2).values,
    'Ann. Vol (%)': (returns.std() * np.sqrt(ann) * 100).round(2).values,
    'Skewness': returns.skew().round(3).values,
    'Excess Kurtosis': (returns.kurtosis()).round(3).values,
    'Min (%)': (returns.min() * 100).round(3).values,
    'Max (%)': (returns.max() * 100).round(3).values,
}).set_index('Asset')

print("✅ Log returns computed")
print(f"   Shape : {returns.shape}")
print("\nSummary Statistics (annualised):")
display(stats)

---
## 📊 Section 3: Exploratory Data Analysis

Before building any portfolio model we need to **understand our data**. We are looking for:

- **Trends and regime shifts** in price levels
- **Volatility clustering** in the return series (variance is not constant — ARCH effects)
- **Fat tails** in return distributions (excess kurtosis — a signature of financial data)
- **Correlation structure** — which assets move together, and whether that structure is stable over time

Each of these observations motivates the modelling choices we make later.

### 3.1 Price Levels — Time Series

All prices are **rebased to 100** on the first observation date so that percentage gains are directly comparable across assets with different price levels.

In [ ]:
# ─────────────────────────────────────────────
# 3.1 Normalised Price Chart
# ─────────────────────────────────────────────

normalised: pd.DataFrame = prices / prices.iloc[0] * 100

fig = go.Figure()

for i, ticker in enumerate(CONFIG['tickers']):
    color = CONFIG['color_palette'][i % len(CONFIG['color_palette'])]
    label = CONFIG['asset_names'].get(ticker, ticker)
    fig.add_trace(
        go.Scatter(
            x=normalised.index,
            y=normalised[ticker],
            name=label,
            mode='lines',
            line=dict(color=color, width=1.8),
            hovertemplate=f'<b>{label}</b><br>Date: %{{x|%Y-%m-%d}}<br>Level: %{{y:.1f}}<extra></extra>',
        )
    )

fig.update_layout(
    title=dict(text='Normalised Price Levels (Base = 100)', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Index Level (Base 100)',
    template=CONFIG['template'],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=480,
    hovermode='x unified',
)
fig.show()

### 3.2 Log Returns — Time Series

Plotting raw return series reveals **volatility clustering** — periods of high volatility (e.g. 2020 COVID crash) are followed by other high-volatility periods, not calm ones. This violates the i.i.d. assumption and motivates GARCH-type models (covered in later episodes).

In [ ]:
# ─────────────────────────────────────────────
# 3.2 Log Returns — Multi-Panel
# ─────────────────────────────────────────────

n_assets = len(CONFIG['tickers'])
fig = make_subplots(
    rows=n_assets,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[CONFIG['asset_names'].get(t, t) for t in CONFIG['tickers']],
    vertical_spacing=0.04,
)

for i, ticker in enumerate(CONFIG['tickers'], start=1):
    color = CONFIG['color_palette'][(i - 1) % len(CONFIG['color_palette'])]
    label = CONFIG['asset_names'].get(ticker, ticker)
    fig.add_trace(
        go.Scatter(
            x=returns.index,
            y=returns[ticker],
            name=label,
            mode='lines',
            line=dict(color=color, width=0.8),
            showlegend=False,
            hovertemplate=f'{label}: %{{y:.4f}}<extra></extra>',
        ),
        row=i, col=1,
    )

fig.update_layout(
    title=dict(text='Daily Log Returns by Asset', font=dict(size=18)),
    template=CONFIG['template'],
    height=180 * n_assets,
    hovermode='x unified',
)
fig.update_xaxes(title_text='Date', row=n_assets, col=1)
fig.show()

### 3.3 Return Distributions — Histograms

Financial returns exhibit **leptokurtosis** (fat tails): extreme events occur far more frequently than a Gaussian distribution predicts. The excess kurtosis for a normal distribution is **0**; empirical equity returns typically show kurtosis of **3–10+**.

This has direct consequences for Value-at-Risk and tail-risk models.

In [ ]:
# ─────────────────────────────────────────────
# 3.3 Return Distributions — Overlaid Histograms
# ─────────────────────────────────────────────

fig = go.Figure()

for i, ticker in enumerate(CONFIG['tickers']):
    color = CONFIG['color_palette'][i % len(CONFIG['color_palette'])]
    label = CONFIG['asset_names'].get(ticker, ticker)
    col_data = returns[ticker].dropna()
    mu = col_data.mean()
    sigma = col_data.std()

    fig.add_trace(
        go.Histogram(
            x=col_data,
            name=label,
            opacity=0.55,
            nbinsx=80,
            marker_color=color,
            hovertemplate=f'{label}<br>Return: %{{x:.4f}}<br>Count: %{{y}}<extra></extra>',
        )
    )

    # ±2σ vertical lines for the first asset only (to avoid clutter)
    if i == 0:
        for mult, label_suffix in [(-2, '-2σ'), (0, 'μ'), (2, '+2σ')]:
            fig.add_vline(
                x=mu + mult * sigma,
                line_dash='dash',
                line_color='white',
                annotation_text=f'{label_suffix}={mu + mult * sigma:.4f}',
                annotation_position='top right',
                opacity=0.6,
            )

fig.update_layout(
    title=dict(text='Daily Log Return Distributions', font=dict(size=18)),
    xaxis_title='Log Return',
    yaxis_title='Frequency',
    barmode='overlay',
    template=CONFIG['template'],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=460,
)
fig.show()

### 3.4 Risk / Return Profile — Box Plots

Box plots show the **full distribution** of daily returns: median, interquartile range, whiskers, and outliers. Comparing boxes side-by-side immediately reveals which assets have the widest spread (highest day-to-day risk).

In [ ]:
# ─────────────────────────────────────────────
# 3.4 Box Plots — Return Distribution per Asset
# ─────────────────────────────────────────────

fig = go.Figure()

for i, ticker in enumerate(CONFIG['tickers']):
    color = CONFIG['color_palette'][i % len(CONFIG['color_palette'])]
    label = CONFIG['asset_names'].get(ticker, ticker)
    fig.add_trace(
        go.Box(
            y=returns[ticker],
            name=label,
            marker_color=color,
            boxmean='sd',
            hovertemplate=f'{label}<br>Return: %{{y:.4f}}<extra></extra>',
        )
    )

fig.update_layout(
    title=dict(text='Daily Return Distributions by Asset (Box Plot)', font=dict(size=18)),
    yaxis_title='Daily Log Return',
    template=CONFIG['template'],
    showlegend=False,
    height=460,
)
fig.show()

### 3.5 Rolling 90-Day Correlation

A **static** correlation number hides how dramatically the relationship between two assets can shift through time. The SPY–TLT (equity–bond) correlation is a textbook example: it was persistently negative during the 2010s (flight-to-quality), but briefly turned positive during the 2020 liquidity crisis.

In [ ]:
# ─────────────────────────────────────────────
# 3.5 Rolling 90-Day Correlation
# ─────────────────────────────────────────────

ROLLING_WINDOW = 90

# Pairs: (ticker_a, ticker_b) — gracefully skip if a ticker is not in the universe
all_tickers = CONFIG['tickers']
pairs: List[Tuple[str, str]] = []
candidate_pairs = [('SPY', 'TLT'), ('SPY', 'GLD'), ('SPY', 'EEM'), ('TLT', 'GLD')]
for a, b in candidate_pairs:
    if a in all_tickers and b in all_tickers:
        pairs.append((a, b))
    if len(pairs) == 3:
        break
if not pairs:  # fallback: first two tickers
    pairs = [(all_tickers[0], all_tickers[1])]

fig = go.Figure()
palette = ['#ef4444', '#3b82f6', '#f59e0b']

for idx, (a, b) in enumerate(pairs):
    label_a = CONFIG['asset_names'].get(a, a)
    label_b = CONFIG['asset_names'].get(b, b)
    pair_label = f'{label_a} / {label_b}'
    rolling_corr = returns[a].rolling(ROLLING_WINDOW).corr(returns[b])
    color = palette[idx % len(palette)]

    fig.add_trace(
        go.Scatter(
            x=rolling_corr.index,
            y=rolling_corr,
            name=pair_label,
            mode='lines',
            line=dict(color=color, width=1.8),
            hovertemplate=f'{pair_label}<br>Date: %{{x|%Y-%m-%d}}<br>ρ = %{{y:.3f}}<extra></extra>',
        )
    )

# Reference lines
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.4,
              annotation_text='ρ = 0', annotation_position='right')
fig.add_hline(y=1, line_dash='dot', line_color='grey', opacity=0.3)
fig.add_hline(y=-1, line_dash='dot', line_color='grey', opacity=0.3)

# Shade crisis period
fig.add_vrect(
    x0=CONFIG['crisis_start'], x1=CONFIG['crisis_end'],
    fillcolor='rgba(239,68,68,0.12)',
    layer='below', line_width=0,
    annotation_text='COVID Crisis', annotation_position='top left',
)

fig.update_layout(
    title=dict(text=f'{ROLLING_WINDOW}-Day Rolling Pairwise Correlation', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Pearson Correlation (ρ)',
    yaxis=dict(range=[-1.1, 1.1]),
    template=CONFIG['template'],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=460,
    hovermode='x unified',
)
fig.show()

### 3.6 Full Correlation Heatmap

The correlation matrix $R$ where $R_{ij} = \rho_{ij}$ gives a complete picture of the pairwise linear relationships across the entire sample period. Values near $-1$ are ideal for diversification; values near $+1$ indicate redundancy.

In [ ]:
# ─────────────────────────────────────────────
# 3.6 Full Correlation Heatmap
# ─────────────────────────────────────────────

corr_matrix: pd.DataFrame = returns.corr()
labels = [CONFIG['asset_names'].get(t, t) for t in corr_matrix.columns]

fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix.values,
        x=labels,
        y=labels,
        colorscale='RdBu',
        zmid=0,
        zmin=-1,
        zmax=1,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont=dict(size=14),
        colorbar=dict(title='ρ'),
        hovertemplate='%{y} / %{x}<br>ρ = %{z:.4f}<extra></extra>',
    )
)

fig.update_layout(
    title=dict(text='Full-Sample Pearson Correlation Matrix', font=dict(size=18)),
    template=CONFIG['template'],
    height=500,
    width=560,
    xaxis=dict(side='bottom'),
)
fig.show()

print("\nCorrelation Matrix:")
display(corr_matrix.round(3))

---
## 📐 Section 4: Portfolio Variance Mathematics

### Full Derivation

**2-asset case:**

$$\sigma^2_p = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \sigma_1 \sigma_2 \rho_{12}$$

**N-asset matrix form:**

$$\sigma^2_p = w^T \Sigma w = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \sigma_{ij}$$

**Decomposition into diagonal and off-diagonal parts:**

$$\sigma^2_p = \underbrace{\sum_{i=1}^{N} w_i^2 \sigma_i^2}_{\text{individual variances}} + \underbrace{\sum_{i \ne j} w_i w_j \sigma_{ij}}_{\text{covariance terms}}$$

**Diversification benefit** (how much variance is eliminated by combining assets):

$$\Delta\sigma^2 = \sum_{i} w_i^2 \sigma_i^2 - \sigma^2_p$$

When all $\rho_{ij} = 1$, diversification benefit $= 0$. When $\rho_{ij} < 0$, off-diagonal covariance terms are negative, so $\Delta\sigma^2 > 0$.

In [ ]:
# ─────────────────────────────────────────────
# 4.1 Covariance Matrix (with optional Ledoit-Wolf shrinkage)
# ─────────────────────────────────────────────

def compute_covariance_matrix(
    returns: pd.DataFrame,
    annualise: bool = True,
    trading_days: int = 252,
    shrink: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Compute the sample and (optionally) Ledoit-Wolf shrunk covariance matrices.

    The Ledoit-Wolf estimator minimises the expected Frobenius-norm loss between
    the estimated and true covariance matrices, reducing estimation error in
    small-sample settings (Ledoit & Wolf, 2004).

    Args:
        returns: DataFrame of daily log returns, shape (T, N).
        annualise: If True, multiply by ``trading_days`` to express in annual terms.
        trading_days: Number of trading days per year used for annualisation.
        shrink: If True, also return the Ledoit-Wolf shrunk estimate.

    Returns:
        Tuple of (sample_cov, shrunk_cov) DataFrames, both shape (N, N).
        If ``shrink=False``, both elements of the tuple are the same sample matrix.
    """
    scale = trading_days if annualise else 1

    # Sample covariance
    sample_cov = returns.cov() * scale

    if not shrink:
        return sample_cov, sample_cov

    # Ledoit-Wolf shrinkage
    lw = LedoitWolf().fit(returns.values)
    shrunk_array = lw.covariance_ * scale
    shrunk_cov = pd.DataFrame(
        shrunk_array,
        index=returns.columns,
        columns=returns.columns,
    )
    alpha = lw.shrinkage_
    print(f"  Ledoit-Wolf shrinkage intensity α = {alpha:.4f}")

    return sample_cov, shrunk_cov


cov_sample, cov_shrunk = compute_covariance_matrix(
    returns,
    trading_days=CONFIG['trading_days_per_year'],
)

print("\n--- Sample Covariance Matrix (annualised) ---")
display(cov_sample.round(6))
print("\n--- Ledoit-Wolf Shrunk Covariance Matrix (annualised) ---")
display(cov_shrunk.round(6))

In [ ]:
# ─────────────────────────────────────────────
# 4.2 Portfolio Metrics
# ─────────────────────────────────────────────

def compute_portfolio_metrics(
    weights: np.ndarray,
    cov_matrix: pd.DataFrame,
    mean_returns: Optional[pd.Series] = None,
    risk_free_rate: float = 0.04,
    trading_days: int = 252,
) -> Dict[str, float]:
    """Compute key risk/return metrics for a given portfolio weight vector.

    Args:
        weights: Array of portfolio weights, shape (N,). Must sum to 1.0.
        cov_matrix: Annualised covariance matrix, shape (N, N).
        mean_returns: Annualised mean returns per asset, shape (N,).
            If None, expected return and Sharpe ratio are not computed.
        risk_free_rate: Annual risk-free rate for Sharpe ratio calculation.
        trading_days: Trading days per year (used only when mean_returns is None
            and we need to annualise from a daily series).

    Returns:
        Dictionary with keys: 'volatility', 'variance', 'expected_return',
        'sharpe_ratio', 'diversification_benefit'.
    """
    w = np.asarray(weights, dtype=float)
    sigma_sq = float(w @ cov_matrix.values @ w)
    sigma = np.sqrt(sigma_sq)

    # Variance contribution from diagonal (individual variances only)
    diag_variances = np.diag(cov_matrix.values)
    weighted_individual_var = float(np.sum(w ** 2 * diag_variances))
    diversification_benefit = weighted_individual_var - sigma_sq

    metrics: Dict[str, float] = {
        'volatility': sigma,
        'variance': sigma_sq,
        'diversification_benefit': diversification_benefit,
        'expected_return': np.nan,
        'sharpe_ratio': np.nan,
    }

    if mean_returns is not None:
        exp_return = float(w @ np.asarray(mean_returns))
        sharpe = (exp_return - risk_free_rate) / sigma if sigma > 0 else np.nan
        metrics['expected_return'] = exp_return
        metrics['sharpe_ratio'] = sharpe

    return metrics


# ── Set up weights ──────────────────────────
n_assets = len(CONFIG['tickers'])
equal_w = np.ones(n_assets) / n_assets
user_w = np.array(CONFIG['weights']) if CONFIG['weights'] is not None else equal_w

ann_mean = returns.mean() * CONFIG['trading_days_per_year']

ew_metrics = compute_portfolio_metrics(equal_w, cov_shrunk, ann_mean)
user_metrics = compute_portfolio_metrics(user_w, cov_shrunk, ann_mean)

results_df = pd.DataFrame({
    'Equal Weight': ew_metrics,
    'User Config': user_metrics,
}).T

results_df['volatility'] = results_df['volatility'].map('{:.2%}'.format)
results_df['variance'] = results_df['variance'].map('{:.6f}'.format)
results_df['expected_return'] = results_df['expected_return'].map('{:.2%}'.format)
results_df['sharpe_ratio'] = results_df['sharpe_ratio'].map('{:.3f}'.format)
results_df['diversification_benefit'] = results_df['diversification_benefit'].map('{:.6f}'.format)

print("✅ Portfolio metrics computed")
display(results_df)

### 4.2 Marginal Contribution to Risk (MCTR)

MCTR answers: *"If I add a small increment of weight to asset $i$, how much does portfolio volatility increase?"*

$$\text{MCTR}_i = \frac{\partial \sigma_p}{\partial w_i} = \frac{(\Sigma w)_i}{\sigma_p}$$

The **component risk contribution** (percentage of total risk) is:

$$\text{CR}_i = \frac{w_i \cdot \text{MCTR}_i}{\sigma_p}$$

Note: $\sum_i \text{CR}_i = 1$ by Euler's homogeneous function theorem — so CRs are a proper decomposition of total variance.

In [ ]:
# ─────────────────────────────────────────────
# 4.3 Marginal Contribution to Risk (MCTR)
# ─────────────────────────────────────────────

def compute_mctr(
    weights: np.ndarray,
    cov_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Compute Marginal and Component Contribution to Risk for each asset.

    MCTR_i = (Σw)_i / σ_p
    Component Risk CR_i = w_i * MCTR_i / σ_p  (sums to 1)

    Args:
        weights: Portfolio weight vector, shape (N,).
        cov_matrix: Annualised covariance matrix, shape (N, N).

    Returns:
        DataFrame with columns ['weight', 'mctr', 'component_risk_pct'],
        indexed by ticker.
    """
    w = np.asarray(weights, dtype=float)
    sigma_p = np.sqrt(w @ cov_matrix.values @ w)
    mctr = (cov_matrix.values @ w) / sigma_p
    component_risk = w * mctr / sigma_p

    return pd.DataFrame({
        'weight': w,
        'mctr': mctr,
        'component_risk_pct': component_risk * 100,
    }, index=cov_matrix.columns)


mctr_df = compute_mctr(equal_w, cov_shrunk)
mctr_df.index = [CONFIG['asset_names'].get(t, t) for t in mctr_df.index]

print("✅ MCTR computed (equal-weight portfolio)")
display(mctr_df.round(4))

# ── Bar chart ──────────────────────────────
fig = go.Figure(
    go.Bar(
        x=mctr_df.index.tolist(),
        y=mctr_df['component_risk_pct'],
        marker_color=CONFIG['color_palette'][:len(mctr_df)],
        text=mctr_df['component_risk_pct'].round(1).astype(str) + '%',
        textposition='outside',
        hovertemplate='%{x}<br>Component Risk: %{y:.2f}%<extra></extra>',
    )
)
fig.update_layout(
    title=dict(text='Component Risk Contribution (%) — Equal Weight Portfolio', font=dict(size=18)),
    yaxis_title='% of Total Portfolio Risk',
    template=CONFIG['template'],
    height=420,
)
fig.show()

### 4.3 Efficient Frontier — Monte Carlo Simulation

The **Efficient Frontier** (Markowitz, 1952) traces the set of portfolios that offer the maximum expected return for each level of risk. We simulate it by generating thousands of random weight vectors and plotting each portfolio's (volatility, return) point, coloured by Sharpe ratio.

In [ ]:
# ─────────────────────────────────────────────
# 4.4 Monte Carlo Efficient Frontier
# ─────────────────────────────────────────────

def monte_carlo_frontier(
    returns: pd.DataFrame,
    cov_matrix: pd.DataFrame,
    n_portfolios: int = 5000,
    risk_free_rate: float = 0.04,
    trading_days: int = 252,
    random_seed: int = 42,
) -> pd.DataFrame:
    """Simulate random portfolios to approximate the mean-variance efficient frontier.

    Generates ``n_portfolios`` random weight vectors by sampling from a Dirichlet
    distribution (uniform over the simplex), then computes annualised return,
    volatility, and Sharpe ratio for each.

    Args:
        returns: Daily log returns DataFrame, shape (T, N).
        cov_matrix: Annualised covariance matrix, shape (N, N).
        n_portfolios: Number of random portfolios to simulate.
        risk_free_rate: Annual risk-free rate for Sharpe ratio.
        trading_days: Trading days per year.
        random_seed: Random seed for reproducibility.

    Returns:
        DataFrame with columns ['volatility', 'return', 'sharpe'] and
        one column per ticker containing the portfolio weights.
    """
    rng = np.random.default_rng(random_seed)
    n_assets = returns.shape[1]
    ann_mean = returns.mean().values * trading_days
    cov_vals = cov_matrix.values

    results = []
    # Dirichlet(1,...,1) = uniform on the simplex
    weight_matrix = rng.dirichlet(np.ones(n_assets), size=n_portfolios)

    for w in weight_matrix:
        port_return = float(w @ ann_mean)
        port_vol = float(np.sqrt(w @ cov_vals @ w))
        sharpe = (port_return - risk_free_rate) / port_vol if port_vol > 0 else np.nan
        row = {'volatility': port_vol, 'return': port_return, 'sharpe': sharpe}
        for j, ticker in enumerate(returns.columns):
            row[ticker] = w[j]
        results.append(row)

    return pd.DataFrame(results)


print("Simulating 5,000 random portfolios...")
frontier_df = monte_carlo_frontier(
    returns, cov_shrunk,
    n_portfolios=5000,
    trading_days=CONFIG['trading_days_per_year'],
)

# ── Identify special portfolios ─────────────
idx_min_vol = frontier_df['volatility'].idxmin()
idx_max_sharpe = frontier_df['sharpe'].idxmax()
min_vol_pt = frontier_df.loc[idx_min_vol]
max_sharpe_pt = frontier_df.loc[idx_max_sharpe]

# ── Plot ────────────────────────────────────
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=frontier_df['volatility'] * 100,
        y=frontier_df['return'] * 100,
        mode='markers',
        marker=dict(
            color=frontier_df['sharpe'],
            colorscale='Viridis',
            size=4,
            opacity=0.6,
            colorbar=dict(title='Sharpe Ratio'),
        ),
        name='Simulated Portfolios',
        hovertemplate='Vol: %{x:.2f}%<br>Return: %{y:.2f}%<br>Sharpe: %{marker.color:.3f}<extra></extra>',
    )
)

# Mark min-vol
fig.add_trace(
    go.Scatter(
        x=[min_vol_pt['volatility'] * 100],
        y=[min_vol_pt['return'] * 100],
        mode='markers+text',
        marker=dict(color='#10b981', size=16, symbol='star'),
        name='Min Variance',
        text=['Min Vol'],
        textposition='top right',
        hovertemplate=f"Min Vol: {min_vol_pt['volatility']*100:.2f}% | Return: {min_vol_pt['return']*100:.2f}%<extra></extra>",
    )
)

# Mark max-Sharpe
fig.add_trace(
    go.Scatter(
        x=[max_sharpe_pt['volatility'] * 100],
        y=[max_sharpe_pt['return'] * 100],
        mode='markers+text',
        marker=dict(color='#f59e0b', size=16, symbol='diamond'),
        name='Max Sharpe',
        text=['Max Sharpe'],
        textposition='top right',
        hovertemplate=f"Max Sharpe: {max_sharpe_pt['sharpe']:.3f} | Vol: {max_sharpe_pt['volatility']*100:.2f}%<extra></extra>",
    )
)

# Equal weight
ew_metrics = compute_portfolio_metrics(equal_w, cov_shrunk, ann_mean)
fig.add_trace(
    go.Scatter(
        x=[ew_metrics['volatility'] * 100],
        y=[ew_metrics['expected_return'] * 100],
        mode='markers+text',
        marker=dict(color='#ef4444', size=14, symbol='circle'),
        name='Equal Weight',
        text=['EW'],
        textposition='top right',
        hovertemplate=f"Equal Weight | Vol: {ew_metrics['volatility']*100:.2f}%<extra></extra>",
    )
)

fig.update_layout(
    title=dict(text='Monte Carlo Efficient Frontier (5,000 Portfolios)', font=dict(size=18)),
    xaxis_title='Annualised Volatility (%)',
    yaxis_title='Annualised Expected Return (%)',
    template=CONFIG['template'],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=520,
)
fig.show()

print(f"\n✅ Frontier simulated")
print(f"   Min-Variance  : Vol={min_vol_pt['volatility']*100:.2f}%  Return={min_vol_pt['return']*100:.2f}%")
print(f"   Max-Sharpe    : Sharpe={max_sharpe_pt['sharpe']:.3f}  Vol={max_sharpe_pt['volatility']*100:.2f}%")

---
## 📉 Section 5: Crisis Simulation — Correlation Instability

One of the most dangerous assumptions in classical portfolio theory is that the covariance matrix is **stationary**. In practice, correlations rise sharply during market crises — a phenomenon documented empirically by Longin & Solnik (2001) and sometimes called **"correlation breakdown"**.

The intuition: in a crisis, investors simultaneously liquidate many asset classes to raise cash, causing previously uncorrelated assets to fall together. The very diversification you relied upon disappears exactly when you need it most.

### Crisis vs Normal Correlation

We split the full return series into:
- **Normal period**: all data *excluding* the crisis window
- **Crisis period**: COVID-19 (Feb–Apr 2020), defined in `CONFIG['crisis_start/end']`

Then we compute the correlation matrix for each sub-period and display them side by side.

Reference: Longin, F., & Solnik, B. (2001). Extreme correlation of international equity markets. *Journal of Finance*, 56(2), 649–676.

In [ ]:
# ─────────────────────────────────────────────
# 5.1 Normal vs Crisis Correlation Matrices
# ─────────────────────────────────────────────

crisis_mask = (
    (returns.index >= CONFIG['crisis_start']) &
    (returns.index <= CONFIG['crisis_end'])
)

returns_crisis  = returns.loc[crisis_mask]
returns_normal  = returns.loc[~crisis_mask]

corr_normal = returns_normal.corr()
corr_crisis = returns_crisis.corr()

print(f"Normal period  : {returns_normal.shape[0]} trading days")
print(f"Crisis period  : {returns_crisis.shape[0]} trading days "
      f"({CONFIG['crisis_start']} → {CONFIG['crisis_end']})")

labels = [CONFIG['asset_names'].get(t, t) for t in corr_normal.columns]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Normal Period Correlation', 'Crisis Period Correlation'],
    horizontal_spacing=0.12,
)

for col_idx, (corr_mat, title) in enumerate(
    [(corr_normal, 'Normal'), (corr_crisis, 'Crisis')], start=1
):
    fig.add_trace(
        go.Heatmap(
            z=corr_mat.values,
            x=labels,
            y=labels,
            colorscale='RdBu',
            zmid=0, zmin=-1, zmax=1,
            text=np.round(corr_mat.values, 2),
            texttemplate='%{text}',
            textfont=dict(size=12),
            showscale=(col_idx == 2),
            colorbar=dict(title='ρ', x=1.02),
            hovertemplate='%{y} / %{x}<br>ρ = %{z:.4f}<extra></extra>',
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title=dict(
        text=f'Correlation Instability: Normal vs Crisis ({CONFIG["crisis_start"]} – {CONFIG["crisis_end"]})',
        font=dict(size=16),
    ),
    template=CONFIG['template'],
    height=480,
)
fig.show()

In [ ]:
# ─────────────────────────────────────────────
# 5.2 Portfolio Stress Test
# ─────────────────────────────────────────────

def stress_test_portfolio(
    weights: np.ndarray,
    normal_returns: pd.DataFrame,
    crisis_returns: pd.DataFrame,
    trading_days: int = 252,
) -> pd.DataFrame:
    """Compare portfolio volatility under normal vs crisis covariance regimes.

    Computes the annualised covariance matrix separately for each sub-period,
    then evaluates portfolio volatility using the same weight vector.

    Args:
        weights: Portfolio weight vector, shape (N,).
        normal_returns: Log returns during the normal (non-crisis) period.
        crisis_returns: Log returns during the crisis period.
        trading_days: Trading days per year for annualisation.

    Returns:
        DataFrame with columns ['Normal Vol', 'Crisis Vol', 'Change (%)']
        indexed by 'Portfolio' and showing per-asset weight alongside.
    """
    w = np.asarray(weights, dtype=float)

    cov_n = normal_returns.cov().values * trading_days
    cov_c = crisis_returns.cov().values * trading_days

    vol_normal = np.sqrt(w @ cov_n @ w)
    vol_crisis = np.sqrt(w @ cov_c @ w)
    pct_change = (vol_crisis / vol_normal - 1) * 100

    rows = []
    tickers = list(normal_returns.columns)
    for i, ticker in enumerate(tickers):
        name = CONFIG['asset_names'].get(ticker, ticker)
        # Asset-level vol in each regime
        vol_n_i = np.sqrt(cov_n[i, i])
        vol_c_i = np.sqrt(cov_c[i, i])
        rows.append({
            'Asset': name,
            'Weight': f"{w[i]:.1%}",
            'Asset Vol (Normal)': f"{vol_n_i:.2%}",
            'Asset Vol (Crisis)': f"{vol_c_i:.2%}",
            'Asset Vol Δ': f"{(vol_c_i/vol_n_i - 1)*100:+.1f}%",
        })

    asset_df = pd.DataFrame(rows).set_index('Asset')

    print("=" * 55)
    print("PORTFOLIO STRESS TEST RESULTS")
    print("=" * 55)
    print(f"  Portfolio Volatility (Normal Period) : {vol_normal:.2%}")
    print(f"  Portfolio Volatility (Crisis Period) : {vol_crisis:.2%}")
    print(f"  Change in Portfolio Volatility       : {pct_change:+.1f}%")
    print("\nPer-Asset Volatility:")
    display(asset_df)

    return asset_df


_ = stress_test_portfolio(
    equal_w, returns_normal, returns_crisis,
    trading_days=CONFIG['trading_days_per_year'],
)

---
## 🌳 Section 6: Hierarchical Risk Parity (HRP)

**Hierarchical Risk Parity** (López de Prado, 2016) is a machine-learning-based portfolio construction method that overcomes two major weaknesses of classical mean-variance optimisation:

1. **Instability** — mean-variance weights are extremely sensitive to small estimation errors in expected returns
2. **Concentration** — unconstrained optimisers tend to produce highly concentrated portfolios

HRP uses **hierarchical clustering** to identify the correlation structure of assets, and then allocates weights using **recursive bisection** — assigning more weight to clusters with lower variance.

### Algorithm — Three Steps

**Step 1: Tree Clustering**
Convert the correlation matrix into a distance matrix:
$$d_{ij} = \sqrt{\frac{1 - \rho_{ij}}{2}}$$
Then apply hierarchical clustering (e.g. single-linkage) to build a dendrogram.

**Step 2: Quasi-Diagonalisation**
Reorder the assets so that those with similar correlation profiles appear adjacent in the covariance matrix. This makes the matrix "as diagonal as possible" without discarding off-diagonal information.

**Step 3: Recursive Bisection**
Split the reordered asset list into two halves. Assign weights inversely proportional to the sub-cluster's intrinsic variance:
$$\tilde{w}_{C_L} = 1 - \frac{\tilde{V}_{C_L}}{\tilde{V}_{C_L} + \tilde{V}_{C_R}}, \qquad \tilde{V}_{C_k} = \left(\sum_{i \in C_k} \frac{1}{\sigma_i^2}\right)^{-1}$$
Recurse until each sub-cluster contains a single asset.

**Reference:** López de Prado, M. (2016). Building diversified portfolios that outperform out-of-sample. *Journal of Portfolio Management*, 42(4), 59–69.

In [ ]:
# ─────────────────────────────────────────────
# 6.1 HRP Implementation
# ─────────────────────────────────────────────

class HierarchicalRiskParity:
    """Hierarchical Risk Parity (HRP) portfolio optimiser.

    Implements the three-step algorithm from López de Prado (2016):

      1. **Tree Clustering** — build a correlation-distance dendrogram.
      2. **Quasi-Diagonalisation** — reorder assets by cluster membership.
      3. **Recursive Bisection** — assign weights via inverse-variance allocation
         at each level of the tree.

    Unlike mean-variance optimisation, HRP requires no expected-return estimates
    and produces diversified weights without explicit constraints.

    Parameters
    ----------
    linkage_method : str
        Hierarchical clustering linkage method passed to
        ``scipy.cluster.hierarchy.linkage``. One of
        ``'single'``, ``'complete'``, ``'average'``, ``'ward'``.

    Attributes
    ----------
    weights_ : dict[str, float]
        Portfolio weights keyed by ticker, populated after calling ``fit``.
    sorted_tickers_ : list[str]
        Asset order after quasi-diagonalisation.
    link_matrix_ : np.ndarray
        Linkage matrix from hierarchical clustering (shape (N-1, 4)).

    References
    ----------
    López de Prado, M. (2016). Building diversified portfolios that outperform
    out-of-sample. *Journal of Portfolio Management*, 42(4), 59–69.
    """

    def __init__(self, linkage_method: str = 'single') -> None:
        """Initialise HRP with the chosen linkage method."""
        self.linkage_method = linkage_method
        self.weights_: Dict[str, float] = {}
        self.sorted_tickers_: List[str] = []
        self.link_matrix_: Optional[np.ndarray] = None

    # ── Step 1: Tree Clustering ─────────────────
    def _tree_clustering(
        self, corr: pd.DataFrame
    ) -> np.ndarray:
        """Convert correlation matrix to distance matrix and cluster.

        Distance metric: $d_{ij} = \\sqrt{(1 - \\rho_{ij}) / 2}$

        Args:
            corr: Pearson correlation matrix, shape (N, N).

        Returns:
            Linkage matrix of shape (N-1, 4) as returned by
            ``scipy.cluster.hierarchy.linkage``.
        """
        dist = np.sqrt((1.0 - corr.values) / 2.0)
        np.fill_diagonal(dist, 0.0)
        dist = np.clip(dist, 0.0, 1.0)  # numerical guard
        condensed = squareform(dist)
        link = linkage(condensed, method=self.linkage_method)
        return link

    # ── Step 2: Quasi-Diagonalisation ─────────
    @staticmethod
    def _quasi_diagonalise(
        link: np.ndarray, n_assets: int
    ) -> List[int]:
        """Reorder assets so that similar assets are adjacent.

        Traverses the linkage tree recursively, replacing each internal node
        with the sorted list of its leaf children. This produces an ordering
        that makes the covariance matrix block-diagonal.

        Args:
            link: Linkage matrix, shape (N-1, 4).
            n_assets: Number of original assets (leaf nodes).

        Returns:
            List of asset indices in quasi-diagonal order.
        """
        # Each item in `clusters` is a list of original leaf indices
        clusters: Dict[int, List[int]] = {i: [i] for i in range(n_assets)}
        for idx, (left, right, _, _) in enumerate(link):
            node_id = n_assets + idx
            left_i, right_i = int(left), int(right)
            clusters[node_id] = clusters.pop(left_i) + clusters.pop(right_i)
        # The last entry is the root
        root_id = n_assets + len(link) - 1
        return clusters[root_id]

    # ── Step 3: Recursive Bisection ─────────────
    def _recursive_bisection(
        self,
        cov: pd.DataFrame,
        sorted_items: List[str],
    ) -> Dict[str, float]:
        """Assign portfolio weights via top-down recursive bisection.

        At each level, splits the current asset list into two halves and
        allocates weight proportionally to the *inverse* of each half's
        cluster variance (variance of an equal-weighted, inverse-vol portfolio).

        Args:
            cov: Annualised covariance matrix indexed/columned by ticker.
            sorted_items: Ordered list of tickers after quasi-diagonalisation.

        Returns:
            Dictionary mapping each ticker to its HRP weight.
        """
        weights: Dict[str, float] = {t: 1.0 for t in sorted_items}
        items_queue: List[List[str]] = [sorted_items]

        while items_queue:
            items_queue_next: List[List[str]] = []
            for cluster in items_queue:
                if len(cluster) <= 1:
                    continue
                mid = len(cluster) // 2
                left_cluster  = cluster[:mid]
                right_cluster = cluster[mid:]

                v_left  = self._cluster_var(cov, left_cluster)
                v_right = self._cluster_var(cov, right_cluster)

                alpha = 1.0 - v_left / (v_left + v_right)  # weight for left

                for t in left_cluster:
                    weights[t] *= alpha
                for t in right_cluster:
                    weights[t] *= (1.0 - alpha)

                if len(left_cluster) > 1:
                    items_queue_next.append(left_cluster)
                if len(right_cluster) > 1:
                    items_queue_next.append(right_cluster)

            items_queue = items_queue_next

        return weights

    @staticmethod
    def _cluster_var(
        cov: pd.DataFrame, cluster: List[str]
    ) -> float:
        """Compute the inverse-variance portfolio variance for a cluster.

        Uses an inverse-volatility weighting scheme within the cluster:
        $\\tilde{V}_C = (\\sum_{i \\in C} 1/\\sigma_i^2)^{-1}$

        Args:
            cov: Full covariance matrix.
            cluster: List of ticker names belonging to this cluster.

        Returns:
            Scalar cluster variance.
        """
        sub_cov = cov.loc[cluster, cluster].values
        inv_diag = 1.0 / np.diag(sub_cov)
        inv_diag /= inv_diag.sum()  # normalise to sum-to-one
        return float(inv_diag @ sub_cov @ inv_diag)

    def fit(self, returns: pd.DataFrame, cov_matrix: pd.DataFrame) -> 'HierarchicalRiskParity':
        """Fit the HRP model and compute portfolio weights.

        Args:
            returns: Daily log returns, shape (T, N).
            cov_matrix: Annualised covariance matrix, shape (N, N).

        Returns:
            self — weights stored in ``self.weights_``.
        """
        corr = returns.corr()
        tickers = list(returns.columns)

        # Step 1
        self.link_matrix_ = self._tree_clustering(corr)

        # Step 2
        sorted_indices = self._quasi_diagonalise(self.link_matrix_, len(tickers))
        self.sorted_tickers_ = [tickers[i] for i in sorted_indices]

        # Step 3
        self.weights_ = self._recursive_bisection(cov_matrix, self.sorted_tickers_)

        return self


hrp = HierarchicalRiskParity(linkage_method=CONFIG['linkage_method'])
hrp.fit(returns, cov_shrunk)

print("✅ HRP fitted")
print(f"   Linkage method : {CONFIG['linkage_method']}")
print(f"   Quasi-diagonal order : {[CONFIG['asset_names'].get(t, t) for t in hrp.sorted_tickers_]}")
print("\nHRP Weights:")
for ticker, w in hrp.weights_.items():
    print(f"  {CONFIG['asset_names'].get(ticker, ticker):20s}  {w:.4f}  ({w:.1%})")

In [ ]:
# ─────────────────────────────────────────────
# 6.2 Dendrogram
# ─────────────────────────────────────────────

labels_ordered = [CONFIG['asset_names'].get(t, t) for t in returns.columns]

fig = ff.create_dendrogram(
    returns.T.values,
    labels=labels_ordered,
    linkagefun=lambda x: linkage(
        squareform(
            np.clip(np.sqrt((1.0 - np.corrcoef(returns.T.values)) / 2.0), 0, 1)
        ),
        method=CONFIG['linkage_method'],
    ),
    colorscale=CONFIG['color_palette'][:4],
)

fig.update_layout(
    title=dict(
        text=f'HRP Dendrogram — {CONFIG["linkage_method"].capitalize()}-Linkage Clustering',
        font=dict(size=18),
    ),
    xaxis_title='Asset',
    yaxis_title='Correlation Distance',
    template=CONFIG['template'],
    height=420,
)
fig.show()

In [ ]:
# ─────────────────────────────────────────────
# 6.3 HRP Weights Bar Chart
# ─────────────────────────────────────────────

hrp_labels = [CONFIG['asset_names'].get(t, t) for t in hrp.weights_]
hrp_vals   = list(hrp.weights_.values())

fig = go.Figure(
    go.Bar(
        x=hrp_labels,
        y=[v * 100 for v in hrp_vals],
        marker_color=CONFIG['color_palette'][:len(hrp_labels)],
        text=[f"{v:.1%}" for v in hrp_vals],
        textposition='outside',
        hovertemplate='%{x}<br>Weight: %{y:.2f}%<extra></extra>',
    )
)

# Equal weight reference line
ew_pct = 100 / len(hrp_labels)
fig.add_hline(
    y=ew_pct, line_dash='dash', line_color='white', opacity=0.5,
    annotation_text=f'Equal weight ({ew_pct:.1f}%)',
    annotation_position='right',
)

fig.update_layout(
    title=dict(text='HRP Portfolio Weights', font=dict(size=18)),
    yaxis_title='Weight (%)',
    template=CONFIG['template'],
    showlegend=False,
    height=420,
)
fig.show()

In [ ]:
# ─────────────────────────────────────────────
# 6.4 Portfolio Comparison
# ─────────────────────────────────────────────

def max_drawdown(weights: np.ndarray, returns: pd.DataFrame) -> float:
    """Compute the maximum drawdown of a portfolio over the sample period.

    The portfolio return series is constructed as a dot product of daily log
    returns with the weight vector. The maximum drawdown is defined as the
    largest peak-to-trough decline in the cumulative return index.

    Args:
        weights: Weight vector, shape (N,).
        returns: Daily log returns DataFrame, shape (T, N).

    Returns:
        Maximum drawdown as a positive decimal (e.g. 0.35 means 35% drawdown).
    """
    w = np.asarray(weights, dtype=float)
    port_returns = returns.values @ w
    cum_returns = np.exp(np.cumsum(port_returns))  # growth-of-$1
    peak = np.maximum.accumulate(cum_returns)
    drawdown = (cum_returns - peak) / peak
    return float(-drawdown.min())


def compare_portfolios(
    returns: pd.DataFrame,
    cov_matrix: pd.DataFrame,
    frontier_df: pd.DataFrame,
    hrp_weights: Dict[str, float],
    trading_days: int = 252,
    risk_free_rate: float = 0.04,
) -> pd.DataFrame:
    """Build a comparison table across Equal Weight, Min Variance, Max Sharpe, and HRP.

    Args:
        returns: Daily log returns DataFrame.
        cov_matrix: Annualised covariance matrix.
        frontier_df: Monte Carlo frontier DataFrame from ``monte_carlo_frontier``.
        hrp_weights: Dictionary of HRP weights keyed by ticker.
        trading_days: Trading days per year.
        risk_free_rate: Annual risk-free rate.

    Returns:
        DataFrame with one row per portfolio and columns for key metrics.
    """
    tickers = list(returns.columns)
    n = len(tickers)
    ann_mean = returns.mean() * trading_days

    # Weight vectors
    w_ew = np.ones(n) / n
    w_mv = frontier_df.loc[frontier_df['volatility'].idxmin(), tickers].values.astype(float)
    w_ms = frontier_df.loc[frontier_df['sharpe'].idxmax(), tickers].values.astype(float)
    w_hrp = np.array([hrp_weights[t] for t in tickers])

    rows = []
    for label, w in [('Equal Weight', w_ew), ('Min Variance', w_mv),
                     ('Max Sharpe', w_ms), ('HRP', w_hrp)]:
        metrics = compute_portfolio_metrics(w, cov_matrix, ann_mean, risk_free_rate)
        mdd = max_drawdown(w, returns)
        row = {
            'Portfolio': label,
            'Exp. Return (%)': round(metrics['expected_return'] * 100, 2),
            'Volatility (%)': round(metrics['volatility'] * 100, 2),
            'Sharpe Ratio': round(metrics['sharpe_ratio'], 3),
            'Max Drawdown (%)': round(mdd * 100, 2),
            'Div. Benefit': round(metrics['diversification_benefit'], 6),
        }
        rows.append(row)

    return pd.DataFrame(rows).set_index('Portfolio')


comparison = compare_portfolios(
    returns, cov_shrunk, frontier_df, hrp.weights_,
    trading_days=CONFIG['trading_days_per_year'],
)

print("✅ Portfolio comparison complete")
display(comparison)

# ── Grouped bar chart ──────────────────────────────
metrics_to_plot = ['Exp. Return (%)', 'Volatility (%)', 'Sharpe Ratio', 'Max Drawdown (%)']
portfolio_names = comparison.index.tolist()
palette_4 = ['#ef4444', '#3b82f6', '#f59e0b', '#10b981']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=metrics_to_plot,
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (r, c), metric in zip(positions, metrics_to_plot):
    for i, portfolio in enumerate(portfolio_names):
        fig.add_trace(
            go.Bar(
                name=portfolio,
                x=[portfolio],
                y=[comparison.loc[portfolio, metric]],
                marker_color=palette_4[i],
                showlegend=(r == 1 and c == 1),
                hovertemplate=f'{portfolio}<br>{metric}: %{{y:.3f}}<extra></extra>',
            ),
            row=r, col=c,
        )

fig.update_layout(
    title=dict(text='Portfolio Comparison: EW vs Min-Vol vs Max-Sharpe vs HRP', font=dict(size=16)),
    template=CONFIG['template'],
    barmode='group',
    height=560,
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1),
)
fig.show()

---
## 📊 Section 7: Summary Dashboard

In [ ]:
# ─────────────────────────────────────────────
# 7. Summary Dashboard
# ─────────────────────────────────────────────

print("=" * 60)
print("QUANTIFAYA — EP01 SUMMARY DASHBOARD")
print("=" * 60)

print(f"\n{'CONFIGURATION':─<40}")
print(f"  Tickers        : {CONFIG['tickers']}")
print(f"  Period         : {CONFIG['start_date']} → {CONFIG['end_date']}")
print(f"  Trading days   : {CONFIG['trading_days_per_year']}")
print(f"  Crisis window  : {CONFIG['crisis_start']} → {CONFIG['crisis_end']}")
print(f"  HRP linkage    : {CONFIG['linkage_method']}")

print(f"\n{'DATASET':─<40}")
print(f"  Rows fetched   : {prices_raw.shape[0]:,}")
print(f"  Rows (clean)   : {prices.shape[0]:,}")
print(f"  Return obs.    : {returns.shape[0]:,}")

print(f"\n{'KEY STATISTICS':─<40}")
display(stats)

print(f"\n{'FULL-SAMPLE CORRELATION MATRIX':─<40}")
display(corr_matrix.round(3))

print(f"\n{'PORTFOLIO COMPARISON':─<40}")
display(comparison)

# ── Recommend best portfolio ────────────────
best_sharpe_port = comparison['Sharpe Ratio'].idxmax()
best_drawdown_port = comparison['Max Drawdown (%)'].idxmin()

print(f"\n{'RECOMMENDATION':─<40}")
print(f"  Highest Sharpe Ratio : {best_sharpe_port} "
      f"(Sharpe = {comparison.loc[best_sharpe_port, 'Sharpe Ratio']:.3f})")
print(f"  Lowest Max Drawdown  : {best_drawdown_port} "
      f"(MDD = {comparison.loc[best_drawdown_port, 'Max Drawdown (%)']:.2f}%)")
print()
print("  Note: Past performance does not guarantee future results.")
print("  This analysis is for educational purposes only.")
print("=" * 60)

---
## References

1. **Markowitz, H. M.** (1952). Portfolio selection. *Journal of Finance*, 7(1), 77–91.

2. **Markowitz, H. M.** (1959). *Portfolio Selection: Efficient Diversification of Investments*. Wiley.

3. **Ledoit, O., & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.

4. **Ledoit, O., & Wolf, M.** (2004). Honey, I shrunk the sample covariance matrix. *Journal of Portfolio Management*, 30(4), 110–119.

5. **López de Prado, M.** (2016). Building diversified portfolios that outperform out-of-sample. *Journal of Portfolio Management*, 42(4), 59–69.

6. **López de Prado, M.** (2018). *Advances in Financial Machine Learning*. Wiley.

7. **Longin, F., & Solnik, B.** (2001). Extreme correlation of international equity markets. *Journal of Finance*, 56(2), 649–676.

8. **Merton, R. C.** (1972). An analytic derivation of the efficient portfolio frontier. *Journal of Financial and Quantitative Analysis*, 7(4), 1851–1872.

9. **Black, F., & Litterman, R.** (1992). Global portfolio optimization. *Financial Analysts Journal*, 48(5), 28–43.

10. **Sharpe, W. F.** (1964). Capital asset prices: A theory of market equilibrium under conditions of risk. *Journal of Finance*, 19(3), 425–442.

11. **Elton, E. J., & Gruber, M. J.** (1973). Estimating the dependence structure of share prices — implications for portfolio selection. *Journal of Finance*, 28(5), 1203–1232.

12. **DeMiguel, V., Garlappi, L., & Uppal, R.** (2009). Optimal versus naive diversification: How inefficient is the 1/N portfolio strategy? *Review of Financial Studies*, 22(5), 1915–1953.

13. **Michaud, R. O.** (1989). The Markowitz optimization enigma: Is 'optimized' optimal? *Financial Analysts Journal*, 45(1), 31–42.

14. **Ang, A., & Bekaert, G.** (2002). International asset allocation with regime shifts. *Review of Financial Studies*, 15(4), 1137–1187.

15. **Christoffersen, P., Errunza, V., Jacobs, K., & Langlois, H.** (2012). Is the potential for international diversification disappearing? A dynamic copula approach. *Review of Financial Studies*, 25(12), 3711–3751.